# 01. OpenAIEmbeddings

In [1]:
from langchain_teddynote import logging
from dotenv import load_dotenv

load_dotenv()
logging.langsmith('CH08-Embeddings')

LangSmith 추적을 시작합니다.
[프로젝트명]
CH08-Embeddings


In [2]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

In [3]:
text = '임베딩 테스트를 하기 위한 샘플 문장입니다.'

In [4]:
query_result = embeddings.embed_query(text)

In [5]:
len(query_result)

1536

In [6]:
query_result[:5]

[-0.007762908935546875,
 0.036712646484375,
 0.01953125,
 -0.0196990966796875,
 0.0172119140625]

In [7]:
doc_result = embeddings.embed_documents(
    [text, text, text, text]
)

In [8]:
doc_result[0][:5]

[-0.00775146484375,
 0.036712646484375,
 0.01953125,
 -0.0196990966796875,
 0.0172119140625]

In [9]:
len(doc_result[0])

1536

In [10]:
embeddings_1024 = OpenAIEmbeddings(model='text-embedding-3-small', dimensions=1024)

len(embeddings_1024.embed_documents([text])[0])

1024

In [11]:
from sklearn.metrics.pairwise import cosine_similarity

sentence1 = '안녕하세요? 반갑습니다.'
sentence2 = '안녕하세요? 반갑습니다!'
sentence3 = '안녕하세요? 만나서 반가워요.'
sentence4 = 'Hi, nice to meet you.'
sentence5 = 'I like to eat apples.'

sentences = [sentence1, sentence2, sentence3, sentence4, sentence5]
embedded_sentences = embeddings_1024.embed_documents(sentences)

In [12]:
def similarity(a, b):
    return cosine_similarity([a], [b])[0][0]

In [13]:
for i, sentence in enumerate(embedded_sentences):
    for j, other_sentence in enumerate(embedded_sentences):
        if i < j:
            print(
                f"[유사도 {similarity(sentence, other_sentence):.4f}] {sentences[i]} \t <=====> \t {sentences[j]}"
            )

[유사도 0.9644] 안녕하세요? 반갑습니다. 	 <=====> 	 안녕하세요? 반갑습니다!
[유사도 0.8423] 안녕하세요? 반갑습니다. 	 <=====> 	 안녕하세요? 만나서 반가워요.
[유사도 0.5043] 안녕하세요? 반갑습니다. 	 <=====> 	 Hi, nice to meet you.
[유사도 0.1363] 안녕하세요? 반갑습니다. 	 <=====> 	 I like to eat apples.
[유사도 0.8185] 안녕하세요? 반갑습니다! 	 <=====> 	 안녕하세요? 만나서 반가워요.
[유사도 0.4791] 안녕하세요? 반갑습니다! 	 <=====> 	 Hi, nice to meet you.
[유사도 0.1320] 안녕하세요? 반갑습니다! 	 <=====> 	 I like to eat apples.
[유사도 0.5164] 안녕하세요? 만나서 반가워요. 	 <=====> 	 Hi, nice to meet you.
[유사도 0.1458] 안녕하세요? 만나서 반가워요. 	 <=====> 	 I like to eat apples.
[유사도 0.2249] Hi, nice to meet you. 	 <=====> 	 I like to eat apples.


# 02. CacheBackedEmbeddings

In [5]:
from langchain.storage import LocalFileStore
from langchain.embeddings import CacheBackedEmbeddings
from langchain_community.vectorstores.faiss import FAISS
embedding = OpenAIEmbeddings()

store = LocalFileStore('./cache/')

In [6]:
cached_embedder = CacheBackedEmbeddings.from_bytes_store(
    underlying_embeddings=embedding,
    document_embedding_cache=store,
    namespace=embedding.model,
)

In [7]:
list(store.yield_keys())

[]

In [9]:
from langchain.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter

raw_documents = TextLoader('../langchain-kr/08-Embeddings/data/appendix-keywords.txt', encoding='utf-8').load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
documents = text_splitter.split_documents(raw_documents)

In [10]:
%time db = FAISS.from_documents(documents, cached_embedder)

CPU times: total: 891 ms
Wall time: 4.2 s


In [11]:
%time db2 = FAISS.from_documents(documents, cached_embedder)

CPU times: total: 31.2 ms
Wall time: 41.9 ms


In [12]:
from langchain.storage import InMemoryByteStore

store = InMemoryByteStore()

cached_embedder = CacheBackedEmbeddings.from_bytes_store(
    embedding, store, namespace=embedding.model
)

# 03. HuggingFaceEmbeddings

In [13]:
import os
import warnings

warnings.filterwarnings('ignore')

os.environ['HF_HOME'] = './cache/'

In [14]:
texts = [
    '안녕, 만나서 반가워.',
    'Langchain simplifies the process of building applications with large language models',
    '랭체인 한국어 튜토리얼은 LangChain의 공식 문서, cookbook 및 다양한 실용 예제를 바탕으로 하여 사용자가 LangChain을 더 쉽고 효과적으로 활용할 수 있도록 구성되어 있습니다.',
    'LangChain은 초거대 언어모델로 애플리케이션을 구축하는 과정을 단순화합니다.',
    'Retrieval-Augmented Generation (RAG) is an effective technique for improving AI responses.'
]

In [22]:
from dotenv import load_dotenv

load_dotenv()

True

In [31]:
from langchain_huggingface.embeddings import HuggingFaceEndpointEmbeddings

model_name = 'intfloat/multilingual-e5-large-instruction'

hf_embeddings = HuggingFaceEndpointEmbeddings(
    model=model_name,
    task='feature-extraction',
    huggingfacehub_api_token=os.environ['HUGGINGFACEHUB_API_TOKEN']
)

In [32]:
%%time
embedded_documents = hf_embeddings.embed_documents(texts)

CPU times: total: 0 ns
Wall time: 237 ms


HfHubHTTPError: 404 Client Error: Not Found for url: https://router.huggingface.co/hf-inference/pipeline/feature-extraction/intfloat/multilingual-e5-large-instruction (Request ID: Root=1-69c7338e-4ee263c0627e8f612c46d0fd;77682c87-a769-4cd5-8797-39091265e94f)

In [33]:
print('[HuggingFace Endpoint Embedding]')
print(f'Model: \t\t{model_name}')
print(f'Dimension: \t{len(embedded_documents[0])}')

[HuggingFace Endpoint Embedding]
Model: 		intfloat/multilingual-e5-large-instruction


NameError: name 'embedded_documents' is not defined

In [34]:
embedded_query = hf_embeddings.embed_query('LangChain에 대해서 알려주세요.')
embedded_query

HfHubHTTPError: 404 Client Error: Not Found for url: https://router.huggingface.co/hf-inference/pipeline/feature-extraction/intfloat/multilingual-e5-large-instruction (Request ID: Root=1-69c7339e-6dc232372ebec7142ed50ee8;e103aa92-6c98-4245-aceb-065570699379)

In [35]:
import numpy as np

np.array(embedded_query) @ np.array(embedded.documents).T

NameError: name 'embedded_query' is not defined